# 1차 실행 -  논문 목록 다운로드

### 수집은 아주 기본 정보만(차피 상세 다운로드 할 때 초록이랑 키워드 다시 받아야 함)

In [2]:
import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import time
from tqdm import tqdm
from dotenv import load_dotenv

# 1. 환경 설정
if os.path.exists('.env'):
    load_dotenv('.env', override=True)
else:
    raise FileNotFoundError(".env 파일이 없습니다.")

API_KEY = os.getenv("KCI_API_KEY")
BASE_URL = "https://open.kci.go.kr/po/openapi/openApiSearch.kci"

# 검색 설정
SEARCH_TERMS = ['AI', 'Artificial Intelligence', '인공지능']
DATE_FROM = "201601"
DATE_TO = "202512"

def fetch_article_ids_by_term(api_key, journal_name, term):
    articles = []
    page = 1
    display_count = 100
    
    while True:
        params = {
            "apiCode": "articleSearch",
            "key": api_key,
            "journal": journal_name,
            "title": term,
            "keyword": term,
            "dateFrom": DATE_FROM,
            "dateTo": DATE_TO,
            "displayCount": display_count,
            "page": page
        }
        
        try:
            response = requests.get(BASE_URL, params=params, timeout=30)
            response.raise_for_status()
            root = ET.fromstring(response.content)

            total_node = root.find(".//{*}total")
            if total_node is None or int(total_node.text) == 0:
                break
                
            total_count = int(total_node.text)
            records = root.findall(".//{*}record")
            
            for record in records:
                article_info = record.find("{*}articleInfo")
                journal_info = record.find("{*}journalInfo")
                if article_info is None: continue

                # 정보 추출
                category = article_info.findtext(".//{*}article-categories", default="").strip()
                title = article_info.findtext(".//{*}article-title", default="").strip()
                article_id = article_info.get("article-id", "")
                
                articles.append({
                    "학술지명": journal_name,
                    "학술분류": category,  # <--- 새로 추가된 부분
                    "논문ID": article_id,
                    "제목": title,
                    "저자": ", ".join([a.text for a in article_info.findall(".//{*}author") if a.text]),
                    "발행연도": journal_info.findtext("{*}pub-year", default="") if journal_info is not None else "",
                    "KCI_URL": article_info.findtext("{*}url", default="")
                })

            if page * display_count >= total_count:
                break
            page += 1
            time.sleep(0.1)

        except Exception as e:
            print(f"\n[오류] {journal_name} - {term} 수집 중: {e}")
            break
            
    return articles

if __name__ == "__main__":
    INPUT_FILE = '법학 기관 목록.csv'
    OUTPUT_FILE = f'KCI_AI_논문ID목록_{DATE_FROM}_{DATE_TO}.csv'

    try:
        df_org = pd.read_csv(INPUT_FILE, encoding='utf-8')
        target_journals = df_org['학술지한글명'].dropna().unique().tolist()
        
        all_results = []
        
        for jnl in tqdm(target_journals, desc="전체 학술지 검색 중"):
            for term in SEARCH_TERMS:
                res = fetch_article_ids_by_term(API_KEY, jnl, term)
                all_results.extend(res)
            
            if all_results:
                # 중복 제거 후 저장
                pd.DataFrame(all_results).drop_duplicates(subset=['논문ID']).to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

        if all_results:
            df_final = pd.DataFrame(all_results).drop_duplicates(subset=['논문ID'])
            df_final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
            print(f"\n✅ 수집 완료! '학술분류' 포함 총 {len(df_final)}건 저장됨.")
        else:
            print("\n❌ 검색 조건에 부합하는 논문이 없습니다.")

    except Exception as e:
        print(f"치명적 오류: {e}")

전체 학술지 검색 중: 100%|██████████| 159/159 [02:38<00:00,  1.01it/s]


✅ 수집 완료! '학술분류' 포함 총 1072건 저장됨.


### 디버그용 셀(실행 불필요)

In [ ]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("KCI_API_KEY")
BASE_URL = "https://open.kci.go.kr/po/openapi/openApiSearch.kci"

# 테스트할 학술지명 하나 지정
test_journal = "법학연구" 

params = {
    "apiCode": "articleSearch",
    "key": API_KEY,
    "journal": test_journal,
    "displayCount": 1,
    "page": 1
}

try:
    response = requests.get(BASE_URL, params=params)
    print("="*50)
    print(f"HTTP 상태 코드: {response.status_code}")
    print("="*50)
    # 응답 내용 전체 출력
    print(response.text)
    print("="*50)
except Exception as e:
    print(f"요청 중 오류 발생: {e}")

# 세부 참고문헌 정보 다운로드

In [3]:
import os
import requests
import pandas as pd
import time
import re
import xml.etree.ElementTree as ET
from tqdm import tqdm
from dotenv import load_dotenv

# 1. 설정 로드
load_dotenv()
API_KEY = os.getenv("KCI_API_KEY")
BASE_URL = "https://open.kci.go.kr/po/openapi/openApiSearch.kci"

INPUT_FILE = 'KCI_AI_논문_기본정보_목록_201601_202512.csv' 
OUTPUT_FILE = 'KCI_AI_논문_상세_및_인용데이터.csv'
SAVE_INTERVAL = 50 

def fetch_article_detail_and_refs(article_id, title, year):
    """
    논문 ID를 이용해 초록, 키워드, 인용 리스트를 가져오며, 
    기존 파일의 제목과 발행연도를 결합합니다.
    """
    params = {"apiCode": "articleDetail", "key": API_KEY, "id": article_id}
    
    try:
        response = requests.get(BASE_URL, params=params, timeout=30)
        response.encoding = 'utf-8'
        root = ET.fromstring(response.content)

        # --- [기본 정보 추출: 초록 및 키워드] ---
        article_info = root.find(".//{*}articleInfo")
        if article_info is None:
            return [], "No Article Info"

        # 초록 추출
        abstract_nodes = article_info.findall(".//{*}abstract")
        abstract_text = ""
        for ab in abstract_nodes:
            if ab.get('lang') == 'original':
                abstract_text = ab.text.strip() if ab.text else ""
                break
        if not abstract_text and abstract_nodes:
            abstract_text = abstract_nodes[0].text.strip() if abstract_nodes[0].text else ""

        # 키워드 추출
        keywords = [k.text.strip() for k in article_info.findall(".//{*}keyword") if k.text]
        keyword_text = ", ".join(keywords)

        # 학술분류 (카테고리)
        category = article_info.findtext(".//{*}article-categories", default="")

        # --- [인용 정보 추출] ---
        raw_xml = response.text
        ref_matches = re.findall(r'<reference([^>]*)>(?:<!\[CDATA\[)?(.*?)(?:\]\]>)?</reference>', raw_xml, re.DOTALL)
        
        results = []
        
        # 기본 정보 (모든 행에 공통으로 들어갈 데이터)
        base_data = {
            "source_id": article_id,
            "title": title,          # 추가됨
            "pub_year": year,        # 추가됨
            "abstract": abstract_text,
            "keywords": keyword_text,
            "category": category
        }

        if not ref_matches:
            # 인용 정보가 없더라도 상세 정보 저장을 위해 한 행 추가
            row = base_data.copy()
            row.update({
                "target_arti_id": "",
                "ref_type": ""
            })
            results.append(row)
        else:
            for attrs, content in ref_matches:
                arti_id_match = re.search(r'arti-id="([^"]*)"', attrs)
                type_name_match = re.search(r'type-name="([^"]*)"', attrs)
                
                row = base_data.copy()
                row.update({
                    "target_arti_id": arti_id_match.group(1) if arti_id_match else "",
                    "ref_type": type_name_match.group(1) if type_name_match else ""
                    # raw_citation은 여기서 제외됨
                })
                results.append(row)
        
        return results, "Success"

    except Exception as e:
        return [], str(e)

# 2. 이어하기 로직
processed_ids = set()
if os.path.exists(OUTPUT_FILE):
    try:
        df_existing = pd.read_csv(OUTPUT_FILE)
        processed_ids = set(df_existing['source_id'].astype(str).unique())
        print(f"✅ 이미 수집된 논문 {len(processed_ids)}건 발견. 이어서 시작합니다.")
    except:
        pass

# 3. 대상 로드 및 필터링
df_articles = pd.read_csv(INPUT_FILE)
# 아직 처리되지 않은 논문들만 필터링
df_to_process = df_articles[~df_articles['논문ID'].astype(str).isin(processed_ids)].copy()

print(f"📊 수집 대상: 총 {len(df_to_process)}건")

# 4. 메인 루프
all_results = []
try:
    # itertuples를 사용하여 행 단위로 안전하게 순회
    for row in tqdm(df_to_process.itertuples(index=False), total=len(df_to_process), desc="상세 데이터 수집 중"):
        aid = str(row.논문ID)
        title = getattr(row, '제목')
        year = getattr(row, '발행연도')

        details, status = fetch_article_detail_and_refs(aid, title, year)
        
        if details:
            all_results.extend(details)
        
        # 주기적 저장
        if len(all_results) >= SAVE_INTERVAL:
            header = not os.path.exists(OUTPUT_FILE)
            pd.DataFrame(all_results).to_csv(OUTPUT_FILE, mode='a', index=False, header=header, encoding='utf-8-sig')
            all_results = [] 
            
        time.sleep(1.0) # API 부하 방지

except KeyboardInterrupt:
    print("\n🛑 중단되었습니다. 현재까지 데이터를 저장합니다.")

# 5. 최종 저장
if all_results:
    header = not os.path.exists(OUTPUT_FILE)
    pd.DataFrame(all_results).to_csv(OUTPUT_FILE, mode='a', index=False, header=header, encoding='utf-8-sig')

print(f"🏁 수집 완료! 파일 확인: {OUTPUT_FILE}")

📊 수집 대상: 총 1072건


상세 데이터 수집 중: 100%|██████████| 1072/1072 [22:22<00:00,  1.25s/it]

🏁 수집 완료! 파일 확인: KCI_AI_논문_상세_및_인용데이터.csv


### 테스트 코드

In [ ]:
import os
import requests
import pandas as pd
import xml.etree.ElementTree as ET
import time
import re  # 정규표현식 추가
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("KCI_API_KEY")
BASE_URL = "https://open.kci.go.kr/po/openapi/openApiSearch.kci"

def fetch_references_ultimate(article_id):
    references = []
    params = {"apiCode": "articleDetail", "key": API_KEY, "id": article_id}
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "application/xml"
    }
    
    try:
        response = requests.get(BASE_URL, params=params, headers=headers, timeout=30)
        response.encoding = 'utf-8'
        raw_xml = response.text
        
        # 1. 기본 체크
        if "<total>0</total>" in raw_xml:
            return [], "데이터 없음(Total 0)"

        # 2. 정규표현식으로 데이터 직접 추출 (가장 강력한 우회 방법)
        # <reference ...> ... </reference> 사이의 내용을 모두 긁어옵니다.
        # CDATA 태그가 있어도 상관없이 매칭합니다.
        ref_matches = re.findall(r'<reference([^>]*)>(?:<!\[CDATA\[)?(.*?)(?:\]\]>)?</reference>', raw_xml, re.DOTALL)
        
        if ref_matches:
            for attrs, content in ref_matches:
                # 속성값에서 arti-id 추출 (예: arti-id="ART00123")
                arti_id_match = re.search(r'arti-id="([^"]*)"', attrs)
                type_name_match = re.search(r'type-name="([^"]*)"', attrs)
                
                references.append({
                    "source_id": article_id,
                    "target_arti_id": arti_id_match.group(1) if arti_id_match else "",
                    "ref_type": type_name_match.group(1) if type_name_match else "",
                    "raw_citation": content.strip()
                })
            return references, "성공"
        
        # 3. 만약 정규표현식마저 실패했다면 응답 내용 확인용 에러 메시지
        snippet = raw_xml.replace('\n', '')[:100]
        return [], f"추출 실패 (응답 앞부분: {snippet}...)"

    except Exception as e:
        return [], f"예외 발생: {str(e)}"

# --- 실행부 ---
INPUT_FILE = '전체_법학_논문목록.csv'
TEST_OUTPUT_FILE = '법학_인용_최종_우회결과.csv'

if not os.path.exists(INPUT_FILE):
    print("파일 없음")
else:
    df_articles = pd.read_csv(INPUT_FILE)
    df_2020 = df_articles[df_articles['발행연도'].astype(str) == '2020']
    
    # 확실히 있는 997번을 포함하여 11개 구성
    test_ids = ["ART002565997"] + [aid for aid in df_2020['논문ID'].unique() if aid != "ART002565997"]
    test_ids = test_ids[:11]

    all_results = []
    print(f"총 {len(test_ids)}개 논문 우회 테스트 시작")

    for aid in tqdm(test_ids):
        refs, status = fetch_references_ultimate(aid)
        if refs:
            all_results.extend(refs)
            print(f" ✅ ID {aid}: {len(refs)}건 수집 완료")
        else:
            print(f" ❌ ID {aid}: {status}")
        
        time.sleep(1.5)

    if all_results:
        pd.DataFrame(all_results).to_csv(TEST_OUTPUT_FILE, index=False, encoding='utf-8-sig')
        print(f"\n결과 저장 완료: {TEST_OUTPUT_FILE}")